In [7]:
import pandas as pd
import os

files = os.listdir("execution_traces")

In [8]:
import pandas as pd
import os

all_dfs = []

for file in files:
    if not file.endswith(".txt"):
        print(f"Skipping {file} as it is not a text file.")
        continue

    df = pd.read_csv(f"execution_traces/{file}", sep="\t")
    df = df[df.status == "COMPLETED"]
    df = df[df.name.str.contains("TRAIN_AND_PREDICT_CV")]
    if df.empty:
        print(f"Skipping {file} as it has no completed relevant traces.")
        continue

    meta = df.name.str.extract(r"\((.*?)\)")[0]  # Extract string inside parentheses
    model_split_gpu = meta.str.split("_", expand=True)

    df["model"] = model_split_gpu[0].str.split(".", expand=True)[0]
    df["split_type"] = model_split_gpu[1]
    df["gpu"] = model_split_gpu[2].str.split(":", expand=True)[1]

    # df.drop(columns=["name", "hash", "task_id", "exit", "submit", "native_id", "status"], inplace=True)

    all_dfs.append(df)

# Combine all into one DataFrame
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(combined_df.head())
else:
    print("No data collected.")

Skipping execution_trace_2025-03-05_09-28-41.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-02_19-12-11.txt as it has no completed relevant traces.
Skipping execution_trace_2025-04-04_14-01-45.txt as it has no completed relevant traces.
Skipping execution_trace_2025-02-26_09-41-50.txt as it has no completed relevant traces.
Skipping execution_trace_2025-04-04_11-36-30.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-05_16-56-13.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-08_01-58-51.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-05_09-29-12.txt as it has no completed relevant traces.
Skipping execution_trace_2025-04-04_14-01-18.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-07_16-47-13.txt as it has no completed relevant traces.
Skipping execution_trace_2025-04-09_17-18-52.txt as it has no completed relevant traces.
Skipping execution_tr

/var/folders/62/nxnmxsmj1q15tr6b3ywgks88r_qppl/T/ipykernel_92790/754527083.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"execution_traces/{file}", sep="\t")


Skipping execution_trace_2025-03-06_16-06-55.txt as it has no completed relevant traces.


/var/folders/62/nxnmxsmj1q15tr6b3ywgks88r_qppl/T/ipykernel_92790/754527083.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"execution_traces/{file}", sep="\t")


Skipping execution_trace_2025-03-06_13-44-40.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-10_13-07-39.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-07_14-48-22.txt as it has no completed relevant traces.
Skipping execution_trace_2025-04-09_06-55-37.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-08_01-54-04.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-05_09-30-29.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-10_17-09-23.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-02_11-19-09.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-10_17-15-27.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-31_10-58-05.txt as it has no completed relevant traces.
Skipping execution_trace_2025-03-08_02-05-31.txt as it has no completed relevant traces.
Skipping execution_tr

In [9]:
combined_df["submit"] = pd.to_datetime(combined_df["submit"])

# Keep all non-GB/RF or non-LPO data untouched
other_df = combined_df[
    ~((combined_df["model"].isin(["GradientBoosting", "RandomForest"])) & (combined_df["split_type"] == "LPO"))
]

# Keep latest 30 LPO runs each for GradientBoosting and RandomForest separately
latest_rf_gb_lpo = (
    combined_df[
        (combined_df["model"].isin(["GradientBoosting", "RandomForest"])) & (combined_df["split_type"] == "LPO")
    ]
    .sort_values(by="submit", ascending=False)
    .groupby("model")
    .head(80)
)

# Combine both back
combined_df = pd.concat([other_df, latest_rf_gb_lpo], ignore_index=True)

In [10]:
import re


# Function to convert realtime strings (e.g., '2m 30s') into seconds
def convert_to_seconds(time_str):
    if pd.isna(time_str):
        return None
    minutes = re.search(r"(\d+)m", time_str)
    seconds = re.search(r"(\d+)s", time_str)
    total_seconds = 0
    if minutes:
        total_seconds += int(minutes.group(1)) * 60
    if seconds:
        total_seconds += int(seconds.group(1))
    return total_seconds


# Apply to your dataframe
combined_df["realtime_sec"] = combined_df["realtime"].apply(convert_to_seconds)


def format_minutes_seconds(seconds):
    if pd.isna(seconds):
        return "–"
    minutes = int(seconds) // 60
    sec = int(seconds) % 60
    return f"{minutes}m {sec}s"


# Convert again using only LPO
subset = combined_df[combined_df["split_type"] == "LPO"]
grouped = subset.groupby("model")["realtime_sec"]

# Stats
mean = grouped.mean()
std = grouped.std()

# Filter models
exclude_models = {"MOLIR", "SingleDrugProteomicsElasticNet"}
mean = mean[~mean.index.isin(exclude_models)]
std = std[mean.index]

# Format
formatted = pd.DataFrame(
    {"LPO": [f"{format_minutes_seconds(m)} ± {format_minutes_seconds(s)}" for m, s in zip(mean, std)]}, index=mean.index
)

# Sort
sort_order = mean.sort_values(ascending=False)
formatted = formatted.loc[sort_order.index]

# Export LaTeX
latex = formatted.to_latex(
    index=True,
    caption="Mean ± standard deviation of runtimes for multi-drug models (LPO setting), shown as minutes and seconds.",
    label="tab:runtime_lpo_minutes",
    column_format="lc",
)

print(latex)

\begin{table}
\caption{Mean ± standard deviation of runtimes for multi-drug models (LPO setting), shown as minutes and seconds.}
\label{tab:runtime_lpo_minutes}
\begin{tabular}{lc}
\toprule
 & LPO \\
model &  \\
\midrule
MultiOmicsNeuralNetwork & 31m 6s ± 16m 21s \\
DIPK & 28m 48s ± 18m 16s \\
SimpleNeuralNetwork & 22m 0s ± 7m 18s \\
RandomForest & 11m 33s ± 6m 9s \\
GradientBoosting & 1m 47s ± 0m 28s \\
SRMF & 1m 41s ± 0m 14s \\
ElasticNet & 1m 35s ± 0m 55s \\
NaiveMeanEffectsPredictor & 0m 17s ± 0m 22s \\
NaiveDrugMeanPredictor & 0m 11s ± 0m 16s \\
SuperFELTR & 0m 9s ± 0m 13s \\
NaiveCellLineMeanPredictor & 0m 7s ± 0m 9s \\
NaivePredictor & 0m 5s ± 0m 3s \\
\bottomrule
\end{tabular}
\end{table}



In [11]:
print(combined_df[(combined_df["model"] == "GradientBoosting") & (combined_df["split_type"] == "LPO")])

        task_id       hash   native_id  \
674307       47  3b/f1d201  2137196_96   
674311       16  11/6bdc01  2137196_93   
674312       64  f5/99bfd8  2137196_92   
674314       29  4c/eb4266  2137196_98   
674315       12  3e/d49aa3  2137196_90   
...         ...        ...         ...   
674412      391  ec/08a62e     1910589   
674413      390  40/b3e043     1910588   
674414      389  bd/8b9a0f     1910587   
674415      388  48/c3cf39     1910586   
674416      387  45/727a24     1910585   

                                                     name     status exit  \
674307  NFCORE_DRUGRESPONSEEVAL:DRUGRESPONSEEVAL:RUN_C...  COMPLETED    0   
674311  NFCORE_DRUGRESPONSEEVAL:DRUGRESPONSEEVAL:RUN_C...  COMPLETED    0   
674312  NFCORE_DRUGRESPONSEEVAL:DRUGRESPONSEEVAL:RUN_C...  COMPLETED    0   
674314  NFCORE_DRUGRESPONSEEVAL:DRUGRESPONSEEVAL:RUN_C...  COMPLETED    0   
674315  NFCORE_DRUGRESPONSEEVAL:DRUGRESPONSEEVAL:RUN_C...  COMPLETED    0   
...                              